In [6]:
from phoenix import compiler
from phoenix.primitive import ordering, simplification, utils
from phoenix import Hamiltonian
from qiskit.converters import circuit_to_dag
import numpy as np

In [2]:
ham = Hamiltonian(['XXXZIYZI', 'YXXZIYYI', 'ZXXZIYZI'], [-0.0125, -0.0125, -0.0125])
ham_, simp_steps = simplification.simplify_hamiltonian(ham)
qc = utils.constr_circuit_from_simp_steps(ham_, simp_steps)
qc.h(0)

In [3]:
qc.draw(fold=-1)

┌───┐                                                                                                       
q_0: ─┤ H ├───────────────────────────────────────────────────────────────────────────────────────────────────────
      └───┘                  ┌──────┐┌──────────────┐┌──────────────┐             ┌──────┐                        
q_1: ────────────────────────┤0     ├┤0             ├┤0             ├─■───────────┤0     ├────────────────────────
     ┌──────┐                │      ││              ││              │ │           │      │                ┌──────┐
q_2: ┤0     ├────────────────┤      ├┤              ├┤              ├─┼───────────┤      ├────────────────┤0     ├
     │      │                │  Cxz ││              ││              │ │           │  Cxz │                │      │
q_3: ┤  Cyy ├────────────────┤      ├┤              ├┤              ├─┼───────────┤      ├────────────────┤  Cyy ├
     │      │┌──────┐┌──────┐│      ││              ││              │ │           │      │┌──────┐┌──────┐│      │
q_4: ┤1     ├┤0     ├┤0     ├┤1     ├┤  Rzx(-0.025) ├┤  Ryy(-0.025) ├─┼───────────┤1     ├┤0     ├┤0     ├┤1     ├
     └──────┘│  Cxx ││      │└──────┘│              ││              │ │           └──────┘│      ││  Cxx │└──────┘
q_5: ────────┤1     ├┤  Cxx ├────────┤              ├┤              ├─┼───────────────────┤  Cxx ├┤1     ├────────
             └──────┘│      │        │              ││              │ │                   │      │└──────┘        
q_6: ────────────────┤1     ├────────┤              ├┤              ├─┼───────────────────┤1     ├────────────────
                     └──────┘        │              ││              │ │ZZ(-0.025)         └──────┘                
q_7: ────────────────────────────────┤1             ├┤1             ├─■───────────────────────────────────────────
                                     └──────────────┘└──────────────┘

In [7]:
left_end = ordering.compute_left_end(qc)
left_end


array([5, 3, 0, 5, 0, 1, 2, 4])

In [8]:
right_end = ordering.compute_right_end(qc)
right_end

array([5, 3, 0, 5, 0, 1, 2, 4])

In [9]:
right_boundary_qubits = set(np.where(right_end == 0)[0])
# block2's left boundary: qubits with left_end == 0 are at the edge  
left_boundary_qubits = set(np.where(left_end == 0)[0])

# Check if cancelled qubits are at the boundaries
# block1_depth_decrease = bool(cancelled_qubits & right_boundary_qubits)
# block2_depth_decrease = bool(cancelled_qubits & left_boundary_qubits)


In [12]:
bool({0,2,4} & right_boundary_qubits)

True

In [16]:
set(tuple(qc.data[:2]))

TypeError: unhashable type: 'qiskit._accelerate.circuit.CircuitInstruction'

In [18]:
set(tuple(qc.data[0].qubits))

{<Qubit register=(8, "q"), index=2>, <Qubit register=(8, "q"), index=4>}

In [20]:
qc.data[0].qubits[1]

<Qubit register=(8, "q"), index=4>

In [6]:
"""
                                                             ┌──────┐
        q_0: ────────────────────────────────────────────────┤0     ├────────
             ┌──────┐                                        │      │
        q_1: ┤0     ├────────────────────────────────────────┤      ├────────
             │      │                        ┌──────┐┌──────┐│      │┌──────┐
        q_2: ┤  Cxx ├────────────────────────┤0     ├┤0     ├┤  Cyy ├┤0     ├
             │      │┌──────┐┌──────┐┌──────┐│  Czz ││      ││      ││      │
        q_3: ┤1     ├┤0     ├┤0     ├┤0     ├┤1     ├┤  Cxx ├┤      ├┤  Czy ├
             └──────┘│      ││      ││      │└──────┘│      ││      ││      │  ==> right_end = [1, 7, 0, 3, 0, 6, 5, 4]
        q_4: ────────┤  Cxx ├┤      ├┤      ├────────┤1     ├┤1     ├┤1     ├
                     │      ││  Cxx ││      │        └──────┘└──────┘└──────┘
        q_5: ────────┤1     ├┤      ├┤  Cxz ├────────────────────────────────
                     └──────┘│      ││      │
        q_6: ────────────────┤1     ├┤      ├────────────────────────────────
                             └──────┘│      │
        q_7: ────────────────────────┤1     ├────────────────────────────────
                                     └──────┘
                                     
"""

'\n                                                             ┌──────┐\n        q_0: ────────────────────────────────────────────────┤0     ├────────\n             ┌──────┐                                        │      │\n        q_1: ┤0     ├────────────────────────────────────────┤      ├────────\n             │      │                        ┌──────┐┌──────┐│      │┌──────┐\n        q_2: ┤  Cxx ├────────────────────────┤0     ├┤0     ├┤  Cyy ├┤0     ├\n             │      │┌──────┐┌──────┐┌──────┐│  Czz ││      ││      ││      │\n        q_3: ┤1     ├┤0     ├┤0     ├┤0     ├┤1     ├┤  Cxx ├┤      ├┤  Czy ├\n             └──────┘│      ││      ││      │└──────┘│      ││      ││      │  ==> right_end = [1, 7, 0, 3, 0, 6, 5, 4]\n        q_4: ────────┤  Cxx ├┤      ├┤      ├────────┤1     ├┤1     ├┤1     ├\n                     │      ││  Cxx ││      │        └──────┘└──────┘└──────┘\n        q_5: ────────┤1     ├┤      ├┤  Cxz ├────────────────────────────────\n                     └─

In [7]:
from qiskit import QuantumCircuit
qc = QuantumCircuit(8)
qc.cx(1,3)
qc.cx(3,5)
qc.cx(3,6)
qc.cx(3,7)
qc.cx(2,3)
qc.cx(2,4)
qc.cx(0,4)
qc.cx(2,4)
qc.cx(1,3)
qc.draw()

q_0: ─────────────────────────────────────■───────
                                          │       
q_1: ──■─────────────────────────────■────┼───────
       │                             │    │       
q_2: ──┼───────────────────■────■────┼────┼────■──
     ┌─┴─┐               ┌─┴─┐  │  ┌─┴─┐  │    │  
q_3: ┤ X ├──■────■────■──┤ X ├──┼──┤ X ├──┼────┼──
     └───┘  │    │    │  └───┘┌─┴─┐└───┘┌─┴─┐┌─┴─┐
q_4: ───────┼────┼────┼───────┤ X ├─────┤ X ├┤ X ├
          ┌─┴─┐  │    │       └───┘     └───┘└───┘
q_5: ─────┤ X ├──┼────┼───────────────────────────
          └───┘┌─┴─┐  │                           
q_6: ──────────┤ X ├──┼───────────────────────────
               └───┘┌─┴─┐                         
q_7: ───────────────┤ X ├─────────────────────────
                    └───┘

In [8]:
instr1 = qc.data[0]
instr2 = qc.data[-1]

In [9]:
instr1 == instr2

True

In [10]:
instr1.qubits

(<Qubit register=(8, "q"), index=1>, <Qubit register=(8, "q"), index=3>)

In [11]:
from phoenix import CNOTEquivCliffordGate
from qiskit.transpiler import passes, PassManager

cxx = CNOTEquivCliffordGate('x', 'x')
cyy = CNOTEquivCliffordGate('y', 'y')
czz = CNOTEquivCliffordGate('z', 'z')

In [12]:
qc = QuantumCircuit(4)
qc.append(cxx, [0,1])
qc.append(cyy, [1,2])
qc.rz(0.1, 0)
qc.append(czz, [0,2])
qc.append(cxx, [1,3])
qc.draw()

┌──────┐┌─────────┐┌──────┐        
q_0: ┤0     ├┤ Rz(0.1) ├┤0     ├────────
     │  Cxx │└─┬──────┬┘│      │┌──────┐
q_1: ┤1     ├──┤0     ├─┤  Czz ├┤0     ├
     └──────┘  │  Cyy │ │      ││      │
q_2: ──────────┤1     ├─┤1     ├┤  Cxx ├
               └──────┘ └──────┘│      │
q_3: ───────────────────────────┤1     ├
                                └──────┘

In [13]:
qc2 = QuantumCircuit(4)

qc2.append(czz, [0,2])
qc2.append(cxx, [1,3])
qc2.rz(0.1, 0)
qc2.append(cyy, [1,2])
qc2.append(cxx, [0,1])

qc2.draw()

┌──────┐┌─────────┐        ┌──────┐
q_0: ┤0     ├┤ Rz(0.1) ├────────┤0     ├
     │      │└─┬──────┬┘┌──────┐│  Cxx │
q_1: ┤  Czz ├──┤0     ├─┤0     ├┤1     ├
     │      │  │      │ │  Cyy │└──────┘
q_2: ┤1     ├──┤  Cxx ├─┤1     ├────────
     └──────┘  │      │ └──────┘        
q_3: ──────────┤1     ├─────────────────
               └──────┘

In [16]:
ordering.cancellation_bonus(
    ordering._extract_tail_cliffs(qc),
    ordering._extract_head_cliffs(qc2),
)

6.0

In [82]:
circuit_to_dag(qc2)

In [86]:
from phoenix.basics import fSwapEquivCliffordGate, CNOTEquivCliffordGate

def _extract_tail_cliff_block(qc: QuantumCircuit) -> list[tuple[str, str, int, int]]:
    """
    从电路末尾提取连续的 2Q Clifford 块, 跳过单比特旋转门。
    单比特旋转门会阻塞其作用的 qubit, 阻止更早的 2Q 门被收集。
    
    Returns:
        list of (pauli_0, pauli_1, q0, q1) tuples
    """
    block = []
    blocked_qubits = set()
    
    for instr in reversed(qc.data):
        gate = instr.operation
        qubits = tuple(qc.find_bit(q).index for q in instr.qubits)
        
        if isinstance(gate, (CNOTEquivCliffordGate, fSwapEquivCliffordGate)):
            # 2Q Clifford: 只有当两个 qubit 都未被阻塞时才收集
            if not (qubits[0] in blocked_qubits or qubits[1] in blocked_qubits):
                block.append((gate.pauli_0, gate.pauli_1, qubits[0], qubits[1]))
            else:
                # 这个门被阻塞了，把它的 qubits 也标记为阻塞（防止更早的门）
                blocked_qubits.update(qubits)
        
        elif gate.num_qubits == 1:
            # 单比特门：阻塞这个 qubit
            blocked_qubits.add(qubits[0])
        
        else:
            # 其他多比特门：阻塞所有涉及的 qubits
            blocked_qubits.update(qubits)
        
        # 所有 qubits 都被阻塞时提前退出
        if len(blocked_qubits) >= qc.num_qubits:
            break
    
    return block


def _extract_head_cliff_block(qc: QuantumCircuit) -> list[tuple[str, str, int, int]]:
    """
    从电路开头提取连续的 2Q Clifford 块，跳过单比特旋转门。
    """
    block = []
    blocked_qubits = set()
    
    for instr in qc.data:
        gate = instr.operation
        qubits = tuple(qc.find_bit(q).index for q in instr.qubits)
        
        if isinstance(gate, (CNOTEquivCliffordGate, fSwapEquivCliffordGate)):
            if not (qubits[0] in blocked_qubits or qubits[1] in blocked_qubits):
                block.append((gate.pauli_0, gate.pauli_1, qubits[0], qubits[1]))
            else:
                blocked_qubits.update(qubits)
        
        elif gate.num_qubits == 1:
            blocked_qubits.add(qubits[0])
        
        else:
            blocked_qubits.update(qubits)
        
        if len(blocked_qubits) >= qc.num_qubits:
            break
    
    return block



def cancellation_bonus(tail_block: list[tuple], head_block: list[tuple]) -> float:
    """
    计算两个 Clifford 块之间的消除奖励，考虑可交换门的重排。
    
    门可以"穿过"与它 qubits 不相交的门移动到边界。
    """
    bonus = 0.0
    used_tail = set()
    used_head = set()
    
    # 贪心匹配：尝试为每个 tail 门找到可消除的 head 门
    for i, t in enumerate(tail_block):
        for j, h in enumerate(head_block):
            if i in used_tail or j in used_head:
                continue
            
            if _can_cancel(t, h):
                # 检查两边是否都可以移动到边界
                if (_is_reachable(tail_block, i, used_tail) and 
                    _is_reachable(head_block, j, used_head)):
                    bonus += 2.0
                    used_tail.add(i)
                    used_head.add(j)
                    break  # t 已匹配，继续下一个
    
    return bonus


def _is_reachable(block: list[tuple], idx: int, used: set) -> bool:
    """
    检查 block[idx] 是否可以移动到边界（穿过前面所有未使用的门）。
    
    条件：idx 之前所有未使用的门的 qubits 必须与 block[idx] 的 qubits 不相交。
    """
    target_qubits = {block[idx][2], block[idx][3]}
    
    for k in range(idx):
        if k not in used:
            other_qubits = {block[k][2], block[k][3]}
            if not target_qubits.isdisjoint(other_qubits):
                return False  # 被阻塞，无法穿过
    
    return True


def _can_cancel(cliff1: tuple, cliff2: tuple) -> bool:
    """检查两个 Clifford 门是否可以消除（互为逆）"""
    p0_1, p1_1, q0_1, q1_1 = cliff1
    p0_2, p1_2, q0_2, q1_2 = cliff2
    
    # 必须作用在相同 qubits
    if (q0_1, q1_1) != (q0_2, q1_2):
        return False
    
    return _are_inverse_paulis(p0_1, p1_1, p0_2, p1_2)


def _are_inverse_paulis(p0_1: str, p1_1: str, p0_2: str, p1_2: str) -> bool:
    """检查两个 CNOTEquivCliffordGate 是否互为逆"""
    pair1 = (p0_1.upper(), p1_1.upper())
    pair2 = (p0_2.upper(), p1_2.upper())
    
    # 自逆门
    self_inverse = {('X','X'), ('Y','Y'), ('Z','Z'), ('Z','X'), ('X','Z')}
    if pair1 == pair2 and pair1 in self_inverse:
        return True
    
    # 互逆对
    inverse_pairs = [(('X','Y'), ('Y','X')), (('Y','Z'), ('Z','Y'))]
    for inv_pair in inverse_pairs:
        if {pair1, pair2} == set(inv_pair):
            return True
    
    return False

In [87]:
_extract_head_cliff_block(qc2)

[('Z', 'Z', 0, 2), ('X', 'X', 1, 3), ('Y', 'Y', 1, 2)]

In [88]:
_extract_tail_cliff_block(qc)

[('X', 'X', 1, 3), ('Z', 'Z', 0, 2), ('Y', 'Y', 1, 2)]

In [ ]:
# def cancellation_bonus(tail_block: list[tuple], head_block: list[tuple]) -> float:
#     """
#     计算两个 Clifford 块之间的消除奖励。
    
#     tail_block: lhs 末尾的 Clifford 块（第一个元素在边界/最外层）
#     head_block: rhs 开头的 Clifford 块（第一个元素在边界/最外层）
    
#     从边界向内逐对匹配，遇到不能消除的门时：
#     - 如果 qubits 重叠，停止（阻塞）
#     - 如果 qubits 不相交，可以跳过继续匹配（可选的激进模式）
#     """
#     bonus = 0.0
    
#     i, j = 0, 0
#     while i < len(tail_block) and j < len(head_block):
#         t_cliff = tail_block[i]
#         h_cliff = head_block[j]
        
#         if _can_cancel(t_cliff, h_cliff):
#             bonus += 2.0
#             i += 1
#             j += 1
#         else:
#             # 检查是否被阻塞
#             t_qubits = {t_cliff[2], t_cliff[3]}
#             h_qubits = {h_cliff[2], h_cliff[3]}
            
#             if not t_qubits.isdisjoint(h_qubits):
#                 # qubits 有重叠，无法交换，停止
#                 break
#             else:
#                 # qubits 不相交，理论上可以交换
#                 # 简单版本：停止；激进版本：可以尝试跳过
#                 break
    
#     return bonus


# def _can_cancel(cliff1: tuple, cliff2: tuple) -> bool:
#     """检查两个 Clifford 门是否可以消除"""
#     p0_1, p1_1, q0_1, q1_1 = cliff1
#     p0_2, p1_2, q0_2, q1_2 = cliff2
    
#     # 必须作用在相同 qubits
#     if (q0_1, q1_1) != (q0_2, q1_2):
#         return False
    
#     # 检查是否为逆关系
#     return _are_inverse_paulis(p0_1, p1_1, p0_2, p1_2)


# def _are_inverse_paulis(p0_1: str, p1_1: str, p0_2: str, p1_2: str) -> bool:
#     """检查两个 CNOTEquivCliffordGate 是否互为逆"""
#     # 自逆门：cxx, cyy, czz, czx, cxz
#     self_inverse = {('X','X'), ('Y','Y'), ('Z','Z'), ('Z','X'), ('X','Z')}
    
#     # 互逆对
#     inverse_pairs = {
#         (('X','Y'), ('Y','X')),
#         (('Y','Z'), ('Z','Y')),
#     }
    
#     pair1 = (p0_1.upper(), p1_1.upper())
#     pair2 = (p0_2.upper(), p1_2.upper())
    
#     # 检查自逆
#     if pair1 == pair2 and pair1 in self_inverse:
#         return True
    
#     # 检查互逆对
#     for inv_pair in inverse_pairs:
#         if {pair1, pair2} == set(inv_pair):
#             return True
    
#     return False

In [91]:
cancellation_bonus(
    _extract_tail_cliff_block(qc),
    _extract_head_cliff_block(qc2),
)

0.0

In [69]:
qc_final = qc.compose(qc2)

In [72]:
pm = PassManager([
    # passes.InverseCancellation([cxx, cyy, czz]),
    # passes.InverseCancellation([cxx, cyy, czz]),
    passes.CommutativeInverseCancellation(matrix_based=True)
])
pm.run(qc_final).draw()

┌──────┐┌─────────┐┌─────────┐┌──────┐
q_0: ┤0     ├┤ Rz(0.1) ├┤ Rz(0.1) ├┤0     ├
     │  Cxx │└─────────┘└─────────┘│  Cxx │
q_1: ┤1     ├──────────────────────┤1     ├
     └──────┘                      └──────┘
q_2: ──────────────────────────────────────
                                           
q_3: ──────────────────────────────────────

In [71]:
qc_final.draw()

┌──────┐┌─────────┐┌──────┐        ┌──────┐┌─────────┐        ┌──────┐
q_0: ┤0     ├┤ Rz(0.1) ├┤0     ├────────┤0     ├┤ Rz(0.1) ├────────┤0     ├
     │  Cxx │└─┬──────┬┘│      │┌──────┐│      │└─┬──────┬┘┌──────┐│  Cxx │
q_1: ┤1     ├──┤0     ├─┤  Czz ├┤0     ├┤  Czz ├──┤0     ├─┤0     ├┤1     ├
     └──────┘  │  Cyy │ │      ││      ││      │  │      │ │  Cyy │└──────┘
q_2: ──────────┤1     ├─┤1     ├┤  Cxx ├┤1     ├──┤  Cxx ├─┤1     ├────────
               └──────┘ └──────┘│      │└──────┘  │      │ └──────┘        
q_3: ───────────────────────────┤1     ├──────────┤1     ├─────────────────
                                └──────┘          └──────┘

In [11]:
ordering._compute_right_end(qc)

array([1, 7, 0, 3, 0, 6, 5, 4])